# VELOOP Rewards — Comprehensive Data Profiling Foundation
**Author:** Senior Data Analyst & Product Analytics Mentor  
**Project:** VELOOP Rewards Data Analyst Internship (Task 01)  
**Execution Scope:** Days 1–10 Engineering Foundation  
**Target Work Period:** 27 October 2026 – 05 November 2026  

---
### Mandatory Data Authenticity Policy Notice
All datasets analyzed in this notebook are **synthetic and simulated**. In compliance with assignment Section 2 and Section 34, none of the metrics, user counts, task counts, or percentages calculated herein represent actual VELOOP company statistics, real user behavior, or commercial revenue.

### Notebook Objectives:
1. Load all supplied datasets from raw Excel storage safely into memory.
2. Inspect schemas, column counts, record counts, and data types.
3. Validate candidate primary keys and test cross-table referential integrity.
4. Analyze missing value distributions and assess domain-specific business logic validity.
5. Profile duplicate records and isolate anomalies injected into the messy audit packs.
6. Analyze temporal date spans and chronological sequencing.
7. Execute business validation rules via `DataValidator`.
8. Document all unavailable datasets and analytical limitations explicitly without fabricating results.


In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Add src to python path for modular import
notebook_dir = Path.cwd()
src_dir = notebook_dir.parent / "src" if (notebook_dir.parent / "src").exists() else notebook_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from data_loader import DataLoader
from dataset_inspector import DatasetInspector
from missing_value_analysis import MissingValueAnalyzer
from duplicate_analysis import DuplicateAnalyzer
from column_summary import ColumnSummaryAnalyzer
from data_validation import DataValidator
from report_utils import ReportUtils

print(f"Environment initialized. Pandas version: {pd.__version__}")


Environment initialized. Pandas version: 2.3.1


In [2]:
# Initialize DataLoader
raw_dir = notebook_dir.parent / "data" / "raw" if (notebook_dir.parent / "data" / "raw").exists() else notebook_dir / "data" / "raw"
loader = DataLoader(raw_data_dir=raw_dir)

# Load core clean datasets
datasets = loader.load_all_clean_datasets()

# Load audit / messy datasets
users_messy = loader.load_users_messy()
events_messy = loader.load_event_messy_audit()

print("Successfully loaded datasets:")
for name, df in datasets.items():
    print(f"  - {name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"  - users_messy: {users_messy.shape[0]:,} rows x {users_messy.shape[1]} columns")
print(f"  - events_messy: {events_messy.shape[0]:,} rows x {events_messy.shape[1]} columns")


Successfully loaded datasets:
  - users: 1,000 rows x 27 columns
  - sessions: 7,913 rows x 12 columns
  - feature_events: 35,704 rows x 13 columns
  - daily_activity: 6,667 rows x 17 columns
  - task_activity: 9,846 rows x 12 columns
  - milestones: 4,799 rows x 6 columns
  - feature_usage: 15 rows x 7 columns
  - users_messy: 1,008 rows x 28 columns
  - events_messy: 35,714 rows x 13 columns


## 1. Dataset Dimensions & Memory Footprint Overview
Examine the high-level metrics for all loaded tables.


In [3]:
summary_records = []
for name, df in datasets.items():
    inspector = DatasetInspector(df, dataset_name=name)
    info = inspector.get_basic_info()
    summary_records.append(info)

summary_df = pd.DataFrame(summary_records)
summary_df


## 2. Schema Inspection & Data Types Profile
Inspect column types, null counts, null percentages, and unique values for each core table.


In [4]:
# Inspect Users Clean schema
users_inspector = DatasetInspector(datasets['users'], dataset_name='users')
users_schema = users_inspector.inspect_columns()
print("=== USERS CLEAN SCHEMA PROFILE ===")
users_schema


=== USERS CLEAN SCHEMA PROFILE ===


In [5]:
# Inspect Sessions & Feature Events schema
sessions_inspector = DatasetInspector(datasets['sessions'], dataset_name='sessions')
print("=== SESSIONS SCHEMA PROFILE ===")
display(sessions_inspector.inspect_columns())

events_inspector = DatasetInspector(datasets['feature_events'], dataset_name='feature_events')
print("=== FEATURE EVENTS SCHEMA PROFILE ===")
display(events_inspector.inspect_columns())


=== SESSIONS SCHEMA PROFILE ===
                 column_name       data_type  ...  unique_count         sample_value
0                 session_id          object  ...          7913          SESS0000001
1                    user_id          object  ...           984          VLUSR000001
2              session_start  datetime64[ns]  ...          7874  2026-08-11 10:30:01
3                session_end  datetime64[ns]  ...          7905  2026-08-11 10:30:46
4   session_duration_seconds           int64  ...           207                   45
5                     device          object  ...             4              Android
6                   platform          object  ...             1                  Web
7                entry_point          object  ...             6         Help/Support
8               exit_feature          object  ...            15              CAPTCHA
9          events_in_session           int64  ...            33                    7
10       features_used_count     

In [6]:
# Inspect Task Activity schema
tasks_inspector = DatasetInspector(datasets['task_activity'], dataset_name='task_activity')
print("=== TASK ACTIVITY SCHEMA PROFILE ===")
tasks_inspector.inspect_columns()


=== TASK ACTIVITY SCHEMA PROFILE ===


## 3. Primary Key Candidate Identification
Verify which columns meet the mathematical requirements of a Primary Key: **Zero Nulls** and **100% Uniqueness**.


In [7]:
for name, df in datasets.items():
    inspector = DatasetInspector(df, dataset_name=name)
    candidates = inspector.identify_potential_primary_keys()
    print(f"Dataset '{name}': Primary Key Candidate(s) -> {candidates}")


Dataset 'users': Primary Key Candidate(s) -> ['user_id', 'email', 'referral_code']
Dataset 'sessions': Primary Key Candidate(s) -> ['session_id']
Dataset 'feature_events': Primary Key Candidate(s) -> ['event_id']
Dataset 'daily_activity': Primary Key Candidate(s) -> []
Dataset 'task_activity': Primary Key Candidate(s) -> ['activity_id']
Dataset 'milestones': Primary Key Candidate(s) -> []
Dataset 'feature_usage': Primary Key Candidate(s) -> ['feature_name', 'unique_users', 'total_events', 'sessions_with_feature', 'sessions_per_user']


## 4. Referential Integrity & Foreign Key Linkage
Test relational consistency across entities to detect any orphan records.


In [8]:
# Test user_id foreign keys against parent Users table
fk_tests = [
    ("Sessions -> Users", datasets['sessions'], "user_id", datasets['users'], "user_id"),
    ("Feature Events -> Users", datasets['feature_events'], "user_id", datasets['users'], "user_id"),
    ("Daily Activity -> Users", datasets['daily_activity'], "user_id", datasets['users'], "user_id"),
    ("Task Activity -> Users", datasets['task_activity'], "user_id", datasets['users'], "user_id"),
    ("Milestones -> Users", datasets['milestones'], "user_id", datasets['users'], "user_id"),
    ("Feature Events -> Sessions", datasets['feature_events'], "session_id", datasets['sessions'], "session_id"),
    ("Task Activity -> Sessions", datasets['task_activity'], "session_id", datasets['sessions'], "session_id"),
    ("Milestones -> Sessions", datasets['milestones'], "session_id", datasets['sessions'], "session_id"),
]

fk_results = []
for test_name, c_df, c_col, p_df, p_col in fk_tests:
    res = DatasetInspector.check_referential_integrity(c_df, c_col, p_df, p_col, relation_name=test_name)
    fk_results.append({
        "Relation": res["relation_name"],
        "Child Records": res["total_child_records"],
        "Child Distinct": res["distinct_child_values"],
        "Parent Distinct": res["distinct_parent_values"],
        "Orphans Count": res["orphan_distinct_count"],
        "Valid FK": res["is_valid_fk"]
    })

pd.DataFrame(fk_results)


## 5. Missing Value Profiling & Domain Logic Evaluation
Evaluate columns containing null values and assess whether missingness aligns with expected domain behavior.


In [9]:
for name in ['users', 'sessions', 'task_activity', 'milestones']:
    analyzer = MissingValueAnalyzer(datasets[name], dataset_name=name)
    missing_df = analyzer.summarize_missing()
    if not missing_df.empty:
        print(f"=== MISSING VALUES IN {name.upper()} ===")
        display(missing_df)


=== MISSING VALUES IN USERS ===
              column_name  ...                                  domain_assessment
0  referral_source_detail  ...  Expected: Organic users do not have a referral...
1     last_login_datetime  ...  Investigate: Inactive or newly registered user...

[2 rows x 5 columns]
=== MISSING VALUES IN SESSIONS ===
    column_name  ...                                  domain_assessment
0  exit_feature  ...  Investigate: Sessions may terminate abruptly o...

[1 rows x 5 columns]
=== MISSING VALUES IN TASK_ACTIVITY ===
         column_name  ...                                  domain_assessment
1     failure_reason  ...  Expected: Successful or pending tasks do not h...
0  task_completed_at  ...  Expected: Incomplete, abandoned, or failed tas...

[2 rows x 5 columns]
=== MISSING VALUES IN MILESTONES ===
  column_name  ...                                  domain_assessment
0  session_id  ...  Defect: Orphan milestones recorded without an ...

[1 rows x 5 columns]


In [10]:
# Analyze conditional missingness of task_completed_at and failure_reason across task_status
task_analyzer = MissingValueAnalyzer(datasets['task_activity'], dataset_name='task_activity')
print("=== MISSING COMPLETION TIMESTAMP BY TASK STATUS ===")
display(task_analyzer.analyze_conditional_missingness('task_status', 'task_completed_at'))

print("=== MISSING FAILURE REASON BY TASK STATUS ===")
display(task_analyzer.analyze_conditional_missingness('task_status', 'failure_reason'))


=== MISSING COMPLETION TIMESTAMP BY TASK STATUS ===
             total_records  null_count  null_pct
task_status                                     
Abandoned           2051.0      2051.0     100.0
Completed           5627.0         0.0       0.0
Failed              1520.0      1520.0     100.0
Pending              648.0       648.0     100.0
=== MISSING FAILURE REASON BY TASK STATUS ===
             total_records  null_count  null_pct
task_status                                     
Abandoned           2051.0         0.0       0.0
Completed           5627.0      5627.0     100.0
Failed              1520.0         0.0       0.0
Pending              648.0       648.0     100.0


## 6. Duplicate Analysis & Clean vs Messy Comparison
Verify full-row duplicates and examine injected anomalies in `Users_Messy_Audit`.


In [11]:
# Check full-row duplicates across clean tables
for name, df in datasets.items():
    analyzer = DuplicateAnalyzer(df, dataset_name=name)
    dup_res = analyzer.find_full_row_duplicates()
    print(f"Dataset '{name}': {dup_res['duplicate_count']} full duplicates")

# Compare Clean vs Messy Users
dup_analyzer = DuplicateAnalyzer(datasets['users'], dataset_name='users')
comp_res = dup_analyzer.compare_clean_vs_messy(datasets['users'], users_messy, id_col='user_id')
print("\n=== CLEAN VS MESSY USERS AUDIT COMPARISON ===")
for k, v in comp_res.items():
    print(f"  {k}: {v}")


Dataset 'users': 0 full duplicates
Dataset 'sessions': 0 full duplicates
Dataset 'feature_events': 0 full duplicates
Dataset 'daily_activity': 0 full duplicates
Dataset 'task_activity': 0 full duplicates
Dataset 'milestones': 0 full duplicates
Dataset 'feature_usage': 0 full duplicates

=== CLEAN VS MESSY USERS AUDIT COMPARISON ===
  clean_total_rows: 1000
  messy_total_rows: 1008
  clean_id_duplicates: 0
  messy_id_duplicates: 8
  injected_duplicates: 8


In [12]:
# Isolate the exact duplicate records in Users_Messy_Audit
messy_dup_analyzer = DuplicateAnalyzer(users_messy, dataset_name='users_messy')
messy_dups = messy_dup_analyzer.check_key_duplicates(['user_id'])
print(f"Total duplicate user_id records in messy dataset: {messy_dups['duplicate_key_count']}")
messy_dups['sample_duplicate_groups'][['user_id', 'full_name', 'email', 'city', 'age', 'registration_datetime', 'data_environment']].head(10)


Total duplicate user_id records in messy dataset: 8


## 7. Temporal & Date Span Profiling
Profile registration timestamps, session start/end, and event activity timelines.


In [13]:
for name in ['users', 'sessions', 'feature_events', 'task_activity']:
    col_analyzer = ColumnSummaryAnalyzer(datasets[name], dataset_name=name)
    dt_summary = col_analyzer.summarize_datetime_columns()
    if not dt_summary.empty:
        print(f"=== TEMPORAL PROFILE: {name.upper()} ===")
        display(dt_summary)


D:\VELOOPS_DATA_ANALYTICS\VELOOP-Product-Analytics\src\column_summary.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(self.df[col].dropna().head(10))
D:\VELOOPS_DATA_ANALYTICS\VELOOP-Product-Analytics\src\column_summary.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(self.df[col].dropna().head(10))
D:\VELOOPS_DATA_ANALYTICS\VELOOP-Product-Analytics\src\column_summary.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(self.df[col].dropna().head(10))
D:\VELOOPS_DATA_ANALYTICS\VELOOP-Product-Analytics\src\column_summary.

## 8. Feature Events & Task Categorical Distributions
Examine distribution of interactions across features, event statuses, and task categories.


In [14]:
# Feature Events Breakdown
print("=== FEATURE EVENTS BY FEATURE NAME ===")
display(datasets['feature_events']['feature_name'].value_counts().to_frame("Event_Count"))

print("=== FEATURE EVENTS BY EVENT STATUS ===")
display(datasets['feature_events']['event_status'].value_counts().to_frame("Status_Count"))

print("=== TASK ACTIVITY BY CATEGORY & STATUS ===")
task_pivot = pd.crosstab(datasets['task_activity']['task_category'], datasets['task_activity']['task_status'], margins=True)
display(task_pivot)


=== FEATURE EVENTS BY FEATURE NAME ===
                 Event_Count
feature_name                
Watch Ads               3126
Missions                2780
Streak Rewards          2734
CAPTCHA                 2733
Tap & Earn              2654
Mining Rewards          2616
Referral                2521
Daily Reward            2499
Level System            2427
Spin the Wheel          2425
Wallet                  2172
Leaderboard             1836
Badges                  1832
Exchange Center         1690
Follow & Earn           1659
=== FEATURE EVENTS BY EVENT STATUS ===
              Status_Count
event_status              
Started              18628
Success              14428
Abandoned             1764
Failed                 884
=== TASK ACTIVITY BY CATEGORY & STATUS ===
task_status       Abandoned  Completed  Failed  Pending   All
task_category                                                
Advertisement           463       1275     310      155  2203
App Task                262        765

## 9. Automated Business Rules & Quality Assertions
Execute automated validation tests across domain models using `DataValidator`.


In [15]:
# Run validation rules
user_val = DataValidator.validate_users(datasets['users'])
sess_val = DataValidator.validate_sessions(datasets['sessions'])
task_val = DataValidator.validate_tasks(datasets['task_activity'])

all_validations = user_val + sess_val + task_val
val_df = pd.DataFrame(all_validations)
print("=== BUSINESS RULES VALIDATION SUMMARY ===")
display(val_df[['entity', 'rule', 'passed', 'violations_count', 'severity', 'description']])


=== BUSINESS RULES VALIDATION SUMMARY ===
           entity  ...                                        description
0           users  ...              User ID must match regex ^VLUSR\d{6}$
1           users  ...  Registration timestamp cannot be after last lo...
2           users  ...              User age must fall between 13 and 100
3           users  ...                  XP balance must never be negative
4        sessions  ...            Session ID must match regex ^SESS\d{7}$
5        sessions  ...  Session start timestamp must be <= session end...
6        sessions  ...  Recorded session duration must match end - sta...
7   task_activity  ...  Task status must be in {'Failed', 'Pending', '...
8   task_activity  ...  Completed tasks must have a valid task_complet...
9   task_activity  ...  Completed tasks in current simulated data shou...
10  task_activity  ...  Failed tasks should specify a non-null failure...

[11 rows x 6 columns]


## 10. Explicit Analytical Limitations & Missing Data Findings

In strict accordance with the **Mandatory Data Authenticity Policy** (Section 2) and **Non-Negotiable Integrity Rules** (Section 34), we record the following empirical limitations:

1. **Absence of Wallet Transaction Ledger:**
   - In `03_Feature_Events`, the `Wallet` feature is limited exclusively to `wallet_viewed` (2,172 events).
   - Zero financial credit, debit, or balance ledger records exist in the provided dataset pack.
   - Consequently, wallet velocity, balance liability, and currency exchange volume cannot be calculated from empirical data and must not be fabricated.

2. **Absence of Dedicated Withdrawals Dataset:**
   - The ~1,200 withdrawal records anticipated in Section 28 are missing from the raw files.
   - Payout requests, gateway fulfillment latency, payment rail fees, and cashout retention cannot be analyzed empirically.

3. **Coupled Task-Reward Logging:**
   - In `05_Task_Activity`, column `reward_status` simply records `'Credited'`, `'Not Credited'`, or `'Pending'`.
   - The exact currency quantum, currency type (VE, XP, Gem, Token), and ledger transaction IDs are absent from the raw operational task records.

4. **Missing Relational Referral Graph:**
   - While `referral_code` and `referral_source_detail` exist in `Users_1000_Clean`, there is no directed edge table linking referrer to referee.
   - Viral K-factor and multi-tier network loops cannot be calculated without the edge dataset.

5. **Lack of Granular Auth & Funnel Telemetry:**
   - Individual steps during registration and onboarding are not emitted as events in `03_Feature_Events`.
   - Only aggregate profile status flags (`email_verification_status`, `onboarding_status`) exist in the Users profile.

---
### Conclusion
The empirical foundation established in this notebook confirms that VELOOP possesses strong telemetry for core gamification and tasks, but requires significant data infrastructure engineering for financial ledgers, withdrawal pipelines, and funnel tracking before production launch.
